- SIMA2: 将具身交互建模为一个基于历史观测和指令的序列决策问题。模型 $\pi_\theta$ 接收图像流、语言指令以及历史动作，输出具体的键盘鼠标操作。
    - 视觉、语言和动作处于同一个 Token Stream 中，允许模型在输出动作前进行 Internal Reasoning (CoT)。
    - $a_t, \text{text\_out}_t \sim \pi_\theta(\cdot \mid v_{t-k:t}, \text{text\_history}, I)$
        - $t$ 代表当前时刻。$k$ 代表上下文窗口的长度。$v_{t-k:t}$ 意味着模型输入仅包含从 $t-k$ 到 $t$ 这段时间内的帧（例如最近的 10 秒或 30 帧）。
- 混合训练策略 (Mixed Training Strategy)：为了防止 Catastrophic Forgetting（灾难性遗忘）并保持 Gemini 原有的推理能力，SIMA 2 使用了混合数据微调：

### bridge data

- Bridge Data 的合成步骤如下：
    - 原材料 (Input)：
        - 一段真实的人类玩游戏的视频（Visual Frames）。
        - 人类当时的操作记录（Keyboard/Mouse Actions）。
        - 当前的任务目标（Instruction，例如“去砍树”）。
    - 合成过程 (The "Bridge")：
        - 把视频和动作喂给一个更强的、不做量化剪裁的 Gemini Pro 模型。
        - Prompt (提问 Gemini Pro)：“作为一个专家玩家，看到这个画面并执行了这些动作，请你解释一下你现在的 内心推理 (Internal Reasoning) 是什么？如果有用户问你话，你会怎么 回答 (Dialogue)？”
- 为什么叫 "Bridge"？ 因为它桥接了高层的语言指令（Language）和底层的像素级动作（Embodied Action），填补了人类数据中缺失的“思维过程”。没有这个过程，AI 就不知道为什么要按 "W" 键，它只知道这个时候该按 "W"。